In [18]:
# Part 1: Warm-up Medical Test Paradox (slides Class-05 example)
def bayes_solver(prior_prob, sensitivity, specificity):
    # Calculate the posterior probability P(Disease | Positive) using Bayes' Theorem
    # sensitivity = P(+|Disease) = True positive rate
    # specificity = P(-|Health) = True negative rate

    # 1. Calculate the complements
    prior_health = 1 - prior_prob
    false_positive_rate = 1 - specificity
    # 2. Calculate total probability of a positive test P(+): Use Law of Total Probability
    p_positive_total = (sensitivity * prior_prob) + (false_positive_rate * prior_health)
    # 3. Apply Bayes' Theorem: P(D|+) = P(+|D)*P(D) / P(+)
    posterior = (sensitivity * prior_prob) / p_positive_total

    return posterior

In [19]:
# Class Example Demo
# population infect rate: 0.1%
# 99% accuracy if you are sick, 95% if you are healthy
result = bayes_solver(0.001, 0.99, 0.95)
print(f"Even with a positive test, there is a {result} chance you are actually sick.")

Even with a positive test, there is a 0.019434628975265003 chance you are actually sick.


In [20]:
# Part 2: Naive Bayes model to solve spam email filter
# Step 1: Data Loading & Pre-processing
# Download the spam.csv file from Canvas to your Local machine, then upload it to Jupyter.
# Make sure spam.csv is in the same folder of your Lab-02 code
import pandas as pd
try:
    df = pd.read_csv("spam.csv", encoding = "latin-1")
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: file not found!")

Dataset loaded successfully.


In [33]:
# Step 2: Data Cleaning
# The dataset has extra columns named Unnamed: 2, etc. due to CSV parsing issues
# We drop them and just keep the first two columns.
df = df.iloc[:, :2]
# Rename columns to more descriptive
df.columns = ['label', 'text']

In [34]:
# Step 3: Encode labels
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
# Show the few rows to verify
print(df.head()) # This is a checkpoint, must show the below results then continue work.
print(df['label'].value_counts())

  label                                               text  label_num
0   ham  Go until jurong point, crazy.. Available only ...          0
1   ham                      Ok lar... Joking wif u oni...          0
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...          1
3   ham  U dun say so early hor... U c already then say...          0
4   ham  Nah I don't think he goes to usf, he lives aro...          0
label
ham     4825
spam     747
Name: count, dtype: int64


In [36]:
# Step 4: Native Bayes Classifier using scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 4.1 Feature Engineering (Text Vectorization)
#     Convert raw text messages/emails into a numerical "Bag of Words" matrix
vectorizer = CountVectorizer()
#     X: The feature matrix (rows = emails, columns = word counts)
#     Y: the target labels (0 = ham, 1 = spam)
X = vectorizer.fit_transform(df['text'])
y = df['label_num']
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())} unique words.")
# It shows Naive Bayes Model using only the frequency of words, not their correlations.

Vocabulary size: 8672 unique words.


In [37]:
# 4.2 Split the data into Training and Testing sets
#     Use 80% for training, and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [38]:
# 4.3 Train the model
model = MultinomialNB()
model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3860., 597.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.01]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 8672)","[[ 0., 0., 1.,..., 0., 9., 1.], [ 8.,23., 0.,..., 1., 0., 0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 8672)","[[-10.98,-10.98,-10.28,...,-10.98, -8.67,-10.28], [ -7.83, -6.85,-10.03,..., -9.34,-10.03,-10.03]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8672


In [41]:
# 4.4 Evaluate the model 
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred) # Compare the predicted result with real labels
print(accuracy)
print(confusion_matrix(y_test, y_pred))
# You can also print the detailed classification report
print(classification_report(y_test, y_pred, target_names = ['Ham', 'Spam']))

0.97847533632287
[[952  13]
 [ 11 139]]
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       965
        Spam       0.91      0.93      0.92       150

    accuracy                           0.98      1115
   macro avg       0.95      0.96      0.95      1115
weighted avg       0.98      0.98      0.98      1115

